# TwoStageMLP: daily PM2.5 at held-out stations

A shared **tabular MLP** encodes each station-day independently. Three heads each receive the shared representation and their corresponding CMAQ source value. The three corrected concentrations sum to the predicted total. This model has no temporal convolutions.

Use your **real station-level PM2.5 CSV** with the columns below. The CSV was not included with the supplied Python reference files. Set `MILES_GUESS_PM25_CSV` to its path, or edit `data.csv_path` in `config/two_stage_mlp_regression_torch.yml`. The bundled Cabauw friction-velocity CSV cannot be used for this task.

The configuration enables evidential regression. We train on one set of stations, select the epoch on a second set and evaluate once on the held-out test stations. The prediction is a daily estimate given the available predictors for that day; the split assesses spatial generalization, not forecasting into future years.

### 1. Load the matching configuration

From the unpacked project, install the Torch and notebook extras in your Python 3.11 environment (`python -m pip install -e ".[torch,notebooks]"`). Start Jupyter in the repository or its `notebooks` directory. The configuration defaults to CPU and specifies the optimizer settings, maximum epochs, patience and layer widths.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from scipy.stats import t as student_t
from mlguess.torch.models import seed_everything
from mlguess.torch.nn_data import load_pm25_data, prepare_pm25
from mlguess.torch.nn_training import build_model, fit_model, predict_frame, regression_metrics

CONFIG_NAME = "two_stage_mlp_regression_torch.yml"
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config" / CONFIG_NAME).is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f"Open this notebook from the project containing config/{CONFIG_NAME}.")
with (ROOT / "config" / CONFIG_NAME).open(encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
assert cfg["model"]["name"] == "TwoStageMLP"
train_cfg = cfg["training"]
if train_cfg["device"] == "cpu":
    torch.set_num_threads(cfg["runtime"]["cpu_threads"])
seed_everything(train_cfg["seed"])  # also seed the initial model weights
print(cfg["model"], train_cfg)

### 2. Load and split station observations

Supply one row per station and day. Use the same concentration units (µg/m³) for the PM2.5 target and CMAQ sources. The three source columns must remain in the reference order; `aod` is a predictor, **not a fourth additive source**.

| Contents | CSV columns |
|---|---|
| Station, day and target | `stcode`, `date` (YYYY-MM-DD), `region5`, `pm25` (`lat` and `lon` are optional metadata) |
| CMAQ source estimates | `cmaq_pm25_bb`, `cmaq_pm25_anthro`, `cmaq_pm25_othr` |
| AOD and weather | `aod`, `wrf_pbl`, `wrf_temp`, `wrf_wspd`, `wrf_rh` |
| Land use and station descriptors | `population`, `elevation`, `builtup`, `rice`, `forest`, `minor_road_length`, `major_road_length` |

The reference split is by station within region, with singleton regions reserved for training. Use enough stations for nonempty training, validation and test partitions. Scalers are fitted on training data only. Inputs must be finite; handle missing predictors using a treatment fitted on training data. Missing target observations are excluded from losses and metrics.

In [ ]:
csv_path = Path(os.environ.get("MILES_GUESS_PM25_CSV", cfg["data"]["csv_path"])).expanduser()
if not csv_path.is_absolute():
    csv_path = ROOT / csv_path
if not csv_path.is_file():
    raise FileNotFoundError(f"Station PM2.5 CSV not found: {csv_path}. Set MILES_GUESS_PM25_CSV "
                            "or edit data.csv_path in the YAML, then rerun this cell.")
split_args = {key: cfg["data"][key] for key in ("seed", "test_size", "val_size")}
train_df, val_df, test_df = load_pm25_data(csv_path, split_val=True, **split_args)
parts = {"training": train_df, "validation": val_df, "test": test_df}
display(pd.DataFrame({s: {"stations": f.stcode.nunique(), "rows": len(f),
                          "observed targets": f.pm25.notna().sum()}
                      for s, f in parts.items()}).T)

### 3. Prepare the inputs and initialize the model

The tabular model takes individual rows `[N, 15]`, with each target shaped `[N, 1]`. The builder also supplies the training source/target scaler statistics. The model sums corrected sources in physical units; `threshold: null` gives each source a physical lower bound of zero.

In [ ]:
prepared = prepare_pm25(train_df, val_df, test_df, model_name=cfg["model"]["name"])
model = build_model(cfg["model"]["name"], cfg["model"]["kwargs"], prepared)
first_group = next(iter(prepared.train.tensors.values()))
print("First training group's input(s) and targets:", [tuple(x.shape) for x in first_group])
print(f"{type(model).__name__}: {sum(p.numel() for p in model.parameters()):,} parameters")

### 4. Train and select the epoch on validation stations

The evidential head learns a Normal–Inverse-Gamma (NIG) distribution for the total target. The `training` section of the configuration controls its evidence regularization. The two-stage objective also uses the reference cosine-diversity penalty on the learned source corrections (`lambda_sim`). The lowest validation-loss state is restored before testing. The notebook saves that state and the fitted scalers and feature order together. For inference on new stations, reload `PM25Preprocessor.load(out / "preprocessor.json")` from `mlguess.torch.nn_data` and use its `transform()` method with the saved weights; do not refit the scalers.

In [ ]:
out = ROOT / cfg["runtime"]["output_dir"]
out.mkdir(parents=True, exist_ok=True)
history = fit_model(model, prepared.train, prepared.validation, **train_cfg,
                    checkpoint_path=out / "best_state.pt")
prepared.save(out / "preprocessor.json")
print(f"Best epoch: {history.best_epoch}; epochs run: {history.epochs_run}")
epochs = np.arange(1, history.epochs_run + 1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(epochs, history.train_loss, label="Training stations")
ax.plot(epochs, history.val_loss, label="Validation stations")
ax.axvline(history.best_epoch, color="0.4", ls=":", label="Selected epoch")
ax.set(xlabel="Epoch", ylabel="Objective")
ax.legend()
fig.tight_layout()
plt.show()

### 5. Evaluate all held-out stations

For each observation the predicted location is $\gamma_n$. In the MILES-GUESS paper's notation the aleatoric variance is $\mathbb{E}[\sigma_n^2]=\beta_n/(\alpha_n-1)$ and the epistemic variance is $\operatorname{Var}(\mu_n)=\beta_n/[\nu_n(\alpha_n-1)]$. We use the full Student-$t$ predictive distribution to calculate a central 95% prediction interval: degrees of freedom $2\alpha_n$, location $\gamma_n$ and scale squared $\beta_n(1+\nu_n)/(\alpha_n\nu_n)$. This interval concerns a single PM2.5 measurement. The collector returns predictions in µg/m³ and variances in (µg/m³)².

In [ ]:
points = predict_frame(model, prepared.test, prepared, batch_size=cfg["runtime"]["prediction_batch_size"],
                       device=train_cfg["device"]).sort_values(["stcode", "date"]).reset_index(drop=True)
metrics = regression_metrics(points, target_col="pm25")
if {"nu", "alpha", "beta"}.issubset(points.columns):
    nu, alpha, beta = (points[k].to_numpy() for k in ("nu", "alpha", "beta"))
    scale = np.sqrt(beta * (1.0 + nu) / (alpha * nu))
    half = student_t.ppf(0.975, df=2.0 * alpha) * scale
    points["lower95"] = points["pred"] - half
    points["upper95"] = points["pred"] + half
    observed = points.loc[np.isfinite(points["pm25"])]
    metrics["coverage95"] = observed["pm25"].between(observed["lower95"], observed["upper95"]).mean()
display(pd.Series(metrics, name="Held-out station evaluation").to_frame())

**Check the source accounting.** `pred` is the total. For each row `s_corrected` is a vector of three concentrations in `prepared.source_names` order. The corresponding `corrected_<source>` columns retain the same numbers in separate fields. These are learned contributions constrained by the observed total. Without independent source-resolved observations, agreement with total PM2.5 does not validate each attribution.

In [ ]:
sources = list(prepared.source_names)
corrected_cols = [f"corrected_{s}" for s in sources]
corrected = np.stack(points["s_corrected"].to_numpy())
np.testing.assert_allclose(corrected, points[corrected_cols].to_numpy(), rtol=1e-5, atol=1e-5)
np.testing.assert_allclose(corrected.sum(axis=1), points["pred"], rtol=1e-5, atol=1e-4)
display(points[["stcode", "date", "pm25", "pred", *corrected_cols]].head())

### 6. Inspect predictions in time and save the test records

Plot at most the first 90 calendar days for the test station with the most observed labels. The time axis preserves missing days as gaps. The station plot is illustrative; the reported metrics use **all** observed labels on the held-out stations.

In [ ]:
station = points.loc[np.isfinite(points.pm25), "stcode"].value_counts().idxmax()
view = points.loc[points.stcode.eq(station)].copy()
view["date"] = pd.to_datetime(view["date"])
view = view.sort_values("date").set_index("date").asfreq("D").iloc[:90]
fig, ax = plt.subplots(figsize=(10, 3.8))
if "lower95" in view:
    ax.fill_between(view.index, view.lower95, view.upper95, color="tab:blue", alpha=0.16,
                    label="95% predictive interval")
ax.plot(view.index, view.pred, color="tab:blue", label="Prediction")
ax.scatter(view.index, view.pm25, s=14, color="black", label="Observed")
ax.set(xlabel="Date", ylabel="PM2.5 (µg/m³)", title=f"Held-out station {station}")
ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(out / "test_station.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.stackplot(view.index, *(view[c].to_numpy() for c in corrected_cols), labels=sources, alpha=0.75)
ax.plot(view.index, view.pred, color="black", lw=1.25, label="Predicted total")
ax.set(xlabel="Date", ylabel="Corrected PM2.5 (µg/m³)", title=f"Source contributions at station {station}")
ax.legend(loc="best", fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(out / "test_station_sources.png", dpi=150)
plt.show()

In [ ]:
# Keep the per-row vector as valid JSON in the CSV, alongside numeric source columns.
export = points.copy()
export["s_corrected"] = [json.dumps(np.asarray(v).tolist()) for v in export["s_corrected"]]
export.to_csv(out / "test_predictions.csv", index=False)
pd.Series(metrics, name="test").to_csv(out / "test_metrics.csv")
print(f"Saved {len(points):,} held-out prediction records in {out}.")